# Ccdproc: getting started

`author` Julien Morin <julien.morin@umontpellier.fr>

`date` 11 Oct 2023

Reference: __[https://ccdproc.readthedocs.io/en/latest/](https://ccdproc.readthedocs.io/en/latest/)__
* load a FITS image w/ ccdproc, display data and header
* use image file collections to select data

In [ ]:
# import and settings
import numpy as np
import matplotlib.pyplot as plt
import astropy.io.fits as pf
from astropy.stats import mad_std
from astropy.visualization import ZScaleInterval
import ccdproc as ccdp

## Working with a single FITS file

In [ ]:
# load science FITS file
path_sci = "raw/Wasp_33-0001.fits"
ccd_sci = ccdp.CCDData.read(path_sci, unit='adu')
# convert to astropy.io.fits.hdu.hdulist.HDUList to use metadata access as in course #1
hdul_sci = ccd_sci.to_hdu()
print(type(hdul_sci))
print(hdul_sci.info())
print('\n')
print('EXPOSURE = ', ccd_sci.meta['EXPOSURE'], ' / ', ccd_sci.meta.comments['EXPOSURE'])
print('FILTER = ', ccd_sci.meta['FILTER'], ' / ', ccd_sci.meta.comments['FILTER'])

In [ ]:
# display image
plt.figure()
interval = ZScaleInterval()
z1, z2 = interval.get_limits(ccd_sci)
plt.imshow(ccd_sci.data, origin='lower', vmin=z1, vmax=z2)
plt.show()

In [ ]:
# display image statistics
npix = ccd_sci.shape[0]*ccd_sci.shape[1]
print('npix    mean    median  std  ')
print('-----------------------------')
print("%7d %5.1f   %5.1f  %5.1f" % (npix, np.mean(ccd_sci.data), np.median(ccd_sci.data), np.std(ccd_sci.data)))

In [ ]:
### ToDo: use your own imstat function

In [ ]:
# compute and display histogram
plt.figure()
plt.hist(ccd_sci.data.ravel(), bins=100, log=True)
plt.show()

In [ ]:
# same focusing on low values
fig, ax = plt.subplots()
plt.hist(ccd_sci.data.ravel(), range=(200,600), bins=100, log=True)
plt.show()

In [ ]:
# define cropped image
ccd_sci_crop =  ccd_sci[-256:,-256:]
print(ccd_sci_crop.shape)

In [ ]:
# display cropped image
plt.figure()
interval = ZScaleInterval()
z1, z2 = interval.get_limits(ccd_sci_crop)
plt.imshow(ccd_sci_crop.data, origin='lower', vmin=z1, vmax=z2)
plt.show()
#imstat(path_sci,ccd_sci_crop)

In [ ]:
# cropped image histogram focusing on low values
plt.figure()
plt.hist(ccd_sci_crop.data.ravel(), range=(200,600), bins=100, log=True)
plt.show()

# same as above but overplot on the full image histogram
ax.hist(ccd_sci_crop.data.ravel(), range=(200,600), bins=100, log=True)
plt.show()

## Working with file collections

In [ ]:
# define an image file collection for the whole raw directory
ifc_raw = ccdp.ImageFileCollection('raw/')

# print various attributes of ifc_raw
print(type(ifc_raw))
print('\n')
print(ifc_raw.location)
print('\n')
print(ifc_raw.files)

In [ ]:
# print various attributes of ifc_raw -- continued
print(ifc_raw.summary['file', 'date-obs', 'imagetyp', 'object', 'exposure'])


In [ ]:
# get filenames matching condition on header keyword
fnamel_light = ifc_raw.files_filtered(imagetyp='Light Frame', include_path=True)
print(type(fnamel_light))
print(fnamel_light)

In [ ]:
# filter collection to create a new one containing only files matching condition on header keyword
ifc_light = ifc_raw.filter(imagetyp='Light Frame')
print(ifc_light.summary['file', 'date-obs', 'imagetyp', 'object', 'exposure'])

In [ ]:
# directly create a collection based on file names
ifc_light_2 = ccdp.ImageFileCollection('raw/', glob_include='Wasp_33-*.fit*')
print(ifc_light_2.summary['file', 'date-obs', 'imagetyp', 'object', 'exposure'])

In [ ]:
### Further work: adapt your imstat function for usage on a collection and print statistics for the Wasp-33 images